<a href="https://colab.research.google.com/github/Nifal123/financial-/blob/main/%2003_Optimization_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Notebook 3 - Portfolio Optimization Engine and Efficient Frontier

Author: Your Name
Date: May 2026

What This Notebook Covers
- Monte Carlo simulation of 10,000 random portfolios
- Mean-Variance Optimization (Markowitz 1952)
- Maximum Sharpe Ratio Portfolio
- Minimum Variance Portfolio
- Risk Parity Portfolio (used by Bridgewater)
- Full Efficient Frontier with Capital Market Line


```
# This is formatted as code
```



In [1]:
!pip install yfinance plotly scipy -q

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

save_path = '/content/drive/MyDrive/portfolio_optimizer/'

daily_returns = pd.read_csv(f'{save_path}daily_returns.csv', index_col=0, parse_dates=True)

RISK_FREE_RATE = 0.05
TRADING_DAYS   = 252
NUM_ASSETS     = len(daily_returns.columns)
ASSET_NAMES    = list(daily_returns.columns)

print("Libraries loaded successfully")
print("Assets:", ASSET_NAMES)
print("Number of assets:", NUM_ASSETS)

Mounted at /content/drive
Libraries loaded successfully
Assets: ['US Equities', 'Intl Equities', 'Long Bonds', 'Agg Bonds', 'Gold', 'Commodities', 'Real Estate']
Number of assets: 7


In [2]:
# Calculate mean returns and covariance matrix
# These are the two inputs every optimizer needs

mean_returns = daily_returns.mean() * TRADING_DAYS
cov_matrix   = daily_returns.cov() * TRADING_DAYS

print("Annualized Mean Returns:")
print(mean_returns.round(4))
print("")
print("Covariance Matrix:")
print(cov_matrix.round(6))

Annualized Mean Returns:
US Equities      0.0125
Intl Equities    0.0556
Long Bonds       0.1039
Agg Bonds        0.0669
Gold             0.1475
Commodities     -0.0126
Real Estate      0.0744
dtype: float64

Covariance Matrix:
               US Equities  Intl Equities  Long Bonds  Agg Bonds      Gold  \
US Equities       0.003777       0.002219    0.003142  -0.000139  0.001892   
Intl Equities     0.002219       0.033015    0.005480   0.014910  0.030609   
Long Bonds        0.003142       0.005480    0.020506   0.006572  0.003070   
Agg Bonds        -0.000139       0.014910    0.006572   0.053053  0.015382   
Gold              0.001892       0.030609    0.003070   0.015382  0.037892   
Commodities       0.007819      -0.004100    0.006709  -0.006354 -0.005619   
Real Estate       0.004057       0.029245    0.005197   0.012745  0.033874   

               Commodities  Real Estate  
US Equities       0.007819     0.004057  
Intl Equities    -0.004100     0.029245  
Long Bonds        0.0

In [3]:


# Core portfolio calculation functions
# These three functions are used in every optimization below

def portfolio_return(weights, mean_returns):
    return np.dot(weights, mean_returns)

def portfolio_volatility(weights, cov_matrix):
    return np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))

def portfolio_sharpe(weights, mean_returns, cov_matrix, risk_free_rate=RISK_FREE_RATE):
    ret = portfolio_return(weights, mean_returns)
    vol = portfolio_volatility(weights, cov_matrix)
    return (ret - risk_free_rate) / vol

print("Portfolio functions defined successfully")
print("")
print("Functions available:")
print("  portfolio_return()     - calculates annualized return")
print("  portfolio_volatility() - calculates annualized volatility")
print("  portfolio_sharpe()     - calculates Sharpe ratio")

Portfolio functions defined successfully

Functions available:
  portfolio_return()     - calculates annualized return
  portfolio_volatility() - calculates annualized volatility
  portfolio_sharpe()     - calculates Sharpe ratio


In [4]:
# Monte Carlo Simulation
# Generate 10,000 random portfolios and plot them
# This creates the cloud of dots on the efficient frontier chart

np.random.seed(42)
NUM_PORTFOLIOS = 10000

mc_returns  = np.zeros(NUM_PORTFOLIOS)
mc_vols     = np.zeros(NUM_PORTFOLIOS)
mc_sharpes  = np.zeros(NUM_PORTFOLIOS)
mc_weights  = np.zeros((NUM_PORTFOLIOS, NUM_ASSETS))

print("Running Monte Carlo simulation...")
print("Generating 10,000 random portfolios...")

for i in range(NUM_PORTFOLIOS):
    # Generate random weights that sum to 1
    w = np.random.random(NUM_ASSETS)
    w = w / np.sum(w)

    mc_weights[i] = w
    mc_returns[i] = portfolio_return(w, mean_returns)
    mc_vols[i]    = portfolio_volatility(w, cov_matrix)
    mc_sharpes[i] = portfolio_sharpe(w, mean_returns, cov_matrix)

print("Monte Carlo simulation complete")
print(f"Return range  : {mc_returns.min()*100:.1f}% to {mc_returns.max()*100:.1f}%")
print(f"Vol range     : {mc_vols.min()*100:.1f}% to {mc_vols.max()*100:.1f}%")
print(f"Sharpe range  : {mc_sharpes.min():.2f} to {mc_sharpes.max():.2f}")

Running Monte Carlo simulation...
Generating 10,000 random portfolios...
Monte Carlo simulation complete
Return range  : 1.8% to 10.9%
Vol range     : 7.7% to 17.0%
Sharpe range  : -0.37 to 0.47


In [5]:
# Maximum Sharpe Ratio Portfolio
# This is the portfolio with the best risk-adjusted return

def maximize_sharpe(mean_returns, cov_matrix, risk_free_rate=RISK_FREE_RATE):

    num_assets  = len(mean_returns)
    args        = (mean_returns, cov_matrix, risk_free_rate)

    # We minimize negative Sharpe (same as maximizing Sharpe)
    def neg_sharpe(weights):
        return -portfolio_sharpe(weights, mean_returns, cov_matrix, risk_free_rate)

    # Constraints: weights must sum to 1
    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})

    # Bounds: each weight between 0% and 100% (no short selling)
    bounds = tuple((0, 1) for _ in range(num_assets))

    # Starting point: equal weights
    x0 = np.array([1/num_assets] * num_assets)

    result = minimize(neg_sharpe, x0, method='SLSQP',
                      bounds=bounds, constraints=constraints)

    return result

max_sharpe_result  = maximize_sharpe(mean_returns, cov_matrix)
max_sharpe_weights = max_sharpe_result.x

ms_return = portfolio_return(max_sharpe_weights, mean_returns)
ms_vol    = portfolio_volatility(max_sharpe_weights, cov_matrix)
ms_sharpe = portfolio_sharpe(max_sharpe_weights, mean_returns, cov_matrix)

print("MAX SHARPE PORTFOLIO")
print("=" * 40)
print(f"Annual Return     : {ms_return*100:.2f}%")
print(f"Annual Volatility : {ms_vol*100:.2f}%")
print(f"Sharpe Ratio      : {ms_sharpe:.3f}")
print("")
print("Asset Weights:")
for asset, weight in zip(ASSET_NAMES, max_sharpe_weights):
    print(f"  {asset:20s}: {weight*100:.1f}%")

MAX SHARPE PORTFOLIO
Annual Return     : 12.63%
Annual Volatility : 12.79%
Sharpe Ratio      : 0.596

Asset Weights:
  US Equities         : 0.0%
  Intl Equities       : 0.0%
  Long Bonds          : 48.7%
  Agg Bonds           : 0.0%
  Gold                : 51.3%
  Commodities         : 0.0%
  Real Estate         : 0.0%


In [6]:
# Minimum Variance Portfolio
# This is the portfolio with the lowest possible risk

def minimize_variance(mean_returns, cov_matrix):

    num_assets  = len(mean_returns)

    def portfolio_var(weights):
        return portfolio_volatility(weights, cov_matrix)

    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds      = tuple((0, 1) for _ in range(num_assets))
    x0          = np.array([1/num_assets] * num_assets)

    result = minimize(portfolio_var, x0, method='SLSQP',
                      bounds=bounds, constraints=constraints)

    return result

min_var_result  = minimize_variance(mean_returns, cov_matrix)
min_var_weights = min_var_result.x

mv_return = portfolio_return(min_var_weights, mean_returns)
mv_vol    = portfolio_volatility(min_var_weights, cov_matrix)
mv_sharpe = portfolio_sharpe(min_var_weights, mean_returns, cov_matrix)

print("MINIMUM VARIANCE PORTFOLIO")
print("=" * 40)
print(f"Annual Return     : {mv_return*100:.2f}%")
print(f"Annual Volatility : {mv_vol*100:.2f}%")
print(f"Sharpe Ratio      : {mv_sharpe:.3f}")
print("")
print("Asset Weights:")
for asset, weight in zip(ASSET_NAMES, min_var_weights):
    print(f"  {asset:20s}: {weight*100:.1f}%")

MINIMUM VARIANCE PORTFOLIO
Annual Return     : 1.94%
Annual Volatility : 5.91%
Sharpe Ratio      : -0.518

Asset Weights:
  US Equities         : 91.0%
  Intl Equities       : 0.0%
  Long Bonds          : 0.9%
  Agg Bonds           : 6.1%
  Gold                : 2.0%
  Commodities         : 0.0%
  Real Estate         : 0.0%


In [7]:
# Risk Parity Portfolio
# Each asset contributes EQUALLY to total portfolio risk
# Made famous by Ray Dalio's Bridgewater All Weather Fund

def risk_parity(cov_matrix):

    num_assets = len(cov_matrix)

    def risk_contribution_error(weights):
        weights     = np.array(weights)
        port_vol    = portfolio_volatility(weights, cov_matrix)
        marginal_rc = np.dot(cov_matrix, weights) / port_vol
        rc          = weights * marginal_rc
        target_rc   = port_vol / num_assets
        return np.sum((rc - target_rc) ** 2)

    constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})
    bounds      = tuple((0.01, 1) for _ in range(num_assets))
    x0          = np.array([1/num_assets] * num_assets)

    result = minimize(risk_contribution_error, x0, method='SLSQP',
                      bounds=bounds, constraints=constraints,
                      options={'maxiter': 1000, 'ftol': 1e-12})

    return result

rp_result  = risk_parity(cov_matrix)
rp_weights = rp_result.x
rp_weights = rp_weights / rp_weights.sum()

rp_return = portfolio_return(rp_weights, mean_returns)
rp_vol    = portfolio_volatility(rp_weights, cov_matrix)
rp_sharpe = portfolio_sharpe(rp_weights, mean_returns, cov_matrix)

print("RISK PARITY PORTFOLIO")
print("=" * 40)
print(f"Annual Return     : {rp_return*100:.2f}%")
print(f"Annual Volatility : {rp_vol*100:.2f}%")
print(f"Sharpe Ratio      : {rp_sharpe:.3f}")
print("")
print("Asset Weights:")
for asset, weight in zip(ASSET_NAMES, rp_weights):
    print(f"  {asset:20s}: {weight*100:.1f}%")

RISK PARITY PORTFOLIO
Annual Return     : 5.00%
Annual Volatility : 8.65%
Sharpe Ratio      : 0.000

Asset Weights:
  US Equities         : 29.4%
  Intl Equities       : 9.8%
  Long Bonds          : 15.0%
  Agg Bonds           : 11.1%
  Gold                : 9.7%
  Commodities         : 17.3%
  Real Estate         : 7.9%


In [8]:
# Efficient Frontier Curve
# Find the minimum variance portfolio for each target return level

target_returns = np.linspace(mc_returns.min(), mc_returns.max(), 100)
frontier_vols  = []

print("Building efficient frontier curve...")

for target in target_returns:
    constraints = (
        {'type': 'eq', 'fun': lambda x: np.sum(x) - 1},
        {'type': 'eq', 'fun': lambda x, t=target: portfolio_return(x, mean_returns) - t}
    )
    bounds = tuple((0, 1) for _ in range(NUM_ASSETS))
    x0     = np.array([1/NUM_ASSETS] * NUM_ASSETS)

    result = minimize(portfolio_volatility, x0,
                      args=(cov_matrix,),
                      method='SLSQP',
                      bounds=bounds,
                      constraints=constraints)

    if result.success:
        frontier_vols.append(result.fun)
    else:
        frontier_vols.append(np.nan)

frontier_vols = np.array(frontier_vols)
print("Efficient frontier built successfully")

Building efficient frontier curve...
Efficient frontier built successfully


In [9]:
# THE MAIN CHART — Efficient Frontier with all portfolios
# This is the most impressive chart in your entire project

fig = go.Figure()

# 1. Monte Carlo scatter cloud
fig.add_trace(go.Scatter(
    x    = mc_vols * 100,
    y    = mc_returns * 100,
    mode = 'markers',
    name = 'Random Portfolios',
    marker = dict(
        size  = 4,
        color = mc_sharpes,
        colorscale = 'Viridis',
        showscale  = True,
        colorbar   = dict(title='Sharpe Ratio'),
        opacity    = 0.6
    ),
    hovertemplate = 'Vol: %{x:.1f}%<br>Return: %{y:.1f}%<extra></extra>'
))

# 2. Efficient Frontier line
valid = ~np.isnan(frontier_vols)
fig.add_trace(go.Scatter(
    x    = frontier_vols[valid] * 100,
    y    = target_returns[valid] * 100,
    mode = 'lines',
    name = 'Efficient Frontier',
    line = dict(color='white', width=3)
))

# 3. Max Sharpe Portfolio
fig.add_trace(go.Scatter(
    x    = [ms_vol * 100],
    y    = [ms_return * 100],
    mode = 'markers+text',
    name = 'Max Sharpe',
    text = ['Max Sharpe'],
    textposition = 'top right',
    marker = dict(size=18, color='red', symbol='star',
                  line=dict(width=2, color='white'))
))

# 4. Min Variance Portfolio
fig.add_trace(go.Scatter(
    x    = [mv_vol * 100],
    y    = [mv_return * 100],
    mode = 'markers+text',
    name = 'Min Variance',
    text = ['Min Variance'],
    textposition = 'top right',
    marker = dict(size=18, color='cyan', symbol='diamond',
                  line=dict(width=2, color='white'))
))

# 5. Risk Parity Portfolio
fig.add_trace(go.Scatter(
    x    = [rp_vol * 100],
    y    = [rp_return * 100],
    mode = 'markers+text',
    name = 'Risk Parity',
    text = ['Risk Parity'],
    textposition = 'top right',
    marker = dict(size=18, color='orange', symbol='triangle-up',
                  line=dict(width=2, color='white'))
))

# 6. Individual assets
asset_rets = mean_returns.values * 100
asset_vols = np.sqrt(np.diag(cov_matrix.values)) * 100

for i, asset in enumerate(ASSET_NAMES):
    fig.add_trace(go.Scatter(
        x    = [asset_vols[i]],
        y    = [asset_rets[i]],
        mode = 'markers+text',
        name = asset,
        text = [asset],
        textposition = 'bottom right',
        marker = dict(size=10, symbol='circle',
                      line=dict(width=1, color='white'))
    ))

fig.update_layout(
    title      = dict(
        text='Multi-Asset Efficient Frontier with Optimal Portfolios (2018-2024)',
        font=dict(size=17)
    ),
    xaxis_title= 'Annualized Volatility (%)',
    yaxis_title= 'Annualized Return (%)',
    height     = 650,
    template   = 'plotly_dark',
    hovermode  = 'closest',
    legend     = dict(orientation='v', x=1.12, y=0.5),
    font       = dict(family='Arial', size=12)
)

fig.show()
print("Efficient Frontier chart complete")

Efficient Frontier chart complete


In [10]:
# Portfolio weights comparison chart
# Shows how each strategy allocates differently

import plotly.graph_objects as go

portfolios = {
    'Max Sharpe'  : max_sharpe_weights,
    'Min Variance': min_var_weights,
    'Risk Parity' : rp_weights,
    'Equal Weight': np.array([1/NUM_ASSETS] * NUM_ASSETS)
}

colors = ['#2196F3','#FF9800','#4CAF50','#9C27B0',
          '#F44336','#00BCD4','#FF5722']

fig = go.Figure()

for asset_idx, asset in enumerate(ASSET_NAMES):
    fig.add_trace(go.Bar(
        name         = asset,
        x            = list(portfolios.keys()),
        y            = [portfolios[p][asset_idx] * 100 for p in portfolios],
        marker_color = colors[asset_idx]
    ))

fig.update_layout(
    title      = dict(
        text='Asset Allocation Comparison Across Strategies',
        font=dict(size=18)
    ),
    xaxis_title= 'Portfolio Strategy',
    yaxis_title= 'Weight (%)',
    barmode    = 'stack',
    height     = 500,
    template   = 'plotly_white',
    legend     = dict(orientation='h', y=-0.2)
)

fig.show()

In [11]:
# Summary comparison table of all strategies

summary = pd.DataFrame({
    'Strategy'       : ['Max Sharpe', 'Min Variance', 'Risk Parity', 'Equal Weight'],
    'Ann. Return %'  : [
        round(ms_return*100, 2),
        round(mv_return*100, 2),
        round(rp_return*100, 2),
        round(portfolio_return(np.array([1/NUM_ASSETS]*NUM_ASSETS), mean_returns)*100, 2)
    ],
    'Volatility %'   : [
        round(ms_vol*100, 2),
        round(mv_vol*100, 2),
        round(rp_vol*100, 2),
        round(portfolio_volatility(np.array([1/NUM_ASSETS]*NUM_ASSETS), cov_matrix)*100, 2)
    ],
    'Sharpe Ratio'   : [
        round(ms_sharpe, 3),
        round(mv_sharpe, 3),
        round(rp_sharpe, 3),
        round(portfolio_sharpe(np.array([1/NUM_ASSETS]*NUM_ASSETS), mean_returns, cov_matrix), 3)
    ]
}).set_index('Strategy')

print("PORTFOLIO STRATEGY COMPARISON")
print("=" * 55)
print(summary.to_string())

summary.to_csv(f'{save_path}optimization_results.csv')
print("")
print("Results saved to Google Drive")
print("Notebook 3 Complete - Next: Notebook 4 Backtesting")

PORTFOLIO STRATEGY COMPARISON
              Ann. Return %  Volatility %  Sharpe Ratio
Strategy                                               
Max Sharpe            12.63         12.79         0.596
Min Variance           1.94          5.91        -0.518
Risk Parity            5.00          8.65         0.000
Equal Weight           6.40         10.69         0.131

Results saved to Google Drive
Notebook 3 Complete - Next: Notebook 4 Backtesting
